In [1]:
import time

In [2]:
start_notebook = time.time()

In [3]:
import warnings
warnings.filterwarnings("ignore")

# 1. Parameters

In [4]:
name_dataset = 'IMDB'
name_model = 'gpt-4o'
mode = 'few'
seed = 1
part = 1

In [5]:
path_open = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/03.Inference/{name_dataset}/post12/df_test_{part}.csv'

In [6]:
path_save = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/03.Inference/{name_dataset}/post13/df_test_{part}.csv'

In [7]:
path_credentials = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/credentials/OPENAI_API_KEY.json'

# 2. Load Environment

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import json
import requests
import pandas as pd
from openai import OpenAI

In [10]:
with open(path_credentials, "r") as f:
    credentials = json.load(f)

In [11]:
OPENAI_API_KEY = credentials["OPENAI_API_KEY"]

In [12]:
client = OpenAI(api_key=OPENAI_API_KEY)

# 3. Functions

In [13]:
def few_shot_prompt(text):

    ex_neg_1 = (
        "If only to avoid making this type of film in the future. "
        "This film is interesting as an experiment but tells no cogent story. "
        "One might feel virtuous for sitting thru it because it touches on so many IMPORTANT issues "
        "but it does so without any discernable motive. The viewer comes away with no new perspectives "
        "(unless one comes up with one while one's mind wanders, as it will invariably do during this pointless film). "
        "One might better spend one's time staring out a window at a tree growing."
    )

    ex_pos_1 = (
        "Zentropa is the most original movie I've seen in years. If you like unique thrillers that are influenced "
        "by film noir, then this is just the right cure for all of those Hollywood summer blockbusters clogging "
        "the theaters these days. Von Trier's follow-ups like Breaking the Waves have gotten more acclaim, but this "
        "is really his best work. It is flashy without being distracting and offers the perfect combination of suspense "
        "and dark humor. It's too bad he decided handheld cameras were the wave of the future. It's hard to say who talked "
        "him away from the style he exhibits here, but it's everyone's loss that he went into his heavily theoretical dogma "
        "direction instead."
    )

    intro = (
        "Classify the sentiment of the following movie review from the IMDB dataset.\n"
        "Respond only with a single digit: 0 if the sentiment is negative, "
        "1 if the sentiment is positive.\n"
        "Return only the digit (no words, no punctuation).\n"
    )

    few_shots = (
        "\nHere are some examples:\n\n"
        "Example 1:\n"
        f"Review: \"{ex_pos_1}\"\n"
        "Label: 1\n\n"
        "Example 2:\n"
        f"Review: \"{ex_neg_1}\"\n"
        "Label: 0\n\n"
    )

    target = (
        "Now classify the following review:\n"
        f"Review: \"{text}\"\n"
        "Label:"
    )

    return intro + few_shots + target

In [14]:
def predict_label(text, prompt):

    try:

        start_time = time.perf_counter()
        first_token_time = None
        output_text = ""

        stream = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            top_p=1.0,
            seed=seed,
            stream=True,
            stream_options={"include_usage": True}
        )

        for chunk in stream:
            if first_token_time is None and len(chunk.choices) > 0 and chunk.choices[0].delta.content:
                first_token_time = time.perf_counter()

            if len(chunk.choices) > 0 and chunk.choices[0].delta.content:
                output_text += chunk.choices[0].delta.content

            if chunk.usage is not None:
                usage = chunk.usage

        end_time = time.perf_counter()

        return {
            "prediction": output_text.strip(),
            "latency_ms": (end_time - start_time) * 1000,
            "ttft_ms": (
                (first_token_time - start_time) * 1000
                if first_token_time else None
            ),
            "input_tokens": usage.prompt_tokens,
            "output_tokens": usage.completion_tokens
        }
    
    except:
        
        return {
            "prediction": '-1',
            "latency_ms": '-',
            "ttft_ms": '-',
            "input_tokens": '-',
            "output_tokens": '-'
        }

# 4. Load Dataset

In [15]:
df = pd.read_csv(path_open)

In [16]:
df.shape

(2500, 36)

In [17]:
pred_label = []
pred_latency = []
pred_ttft = []
pred_input_tokens = []
pred_output_tokens = []

In [18]:
for i in range(len(df)):

  text = df['text'].iloc[i]
  prompt = few_shot_prompt(text)
  output = predict_label(text, prompt)

  pred_label.append(int(output['prediction'][0]))
  pred_latency.append(output['latency_ms'])
  pred_ttft.append(output['ttft_ms'])
  pred_input_tokens.append(output['input_tokens'])
  pred_output_tokens.append(output['output_tokens'])

  if (i % 10) == 0:
    print(i)

0


10


20


30


40


...

2450


2460


2470


2480


2490


In [19]:
df[f'{name_model}-{mode}-seed-{seed}-label'] = pred_label
df[f'{name_model}-{mode}-seed-{seed}-latency'] = pred_latency
df[f'{name_model}-{mode}-seed-{seed}-ttft'] = pred_ttft
df[f'{name_model}-{mode}-seed-{seed}-input-tokens'] = pred_input_tokens
df[f'{name_model}-{mode}-seed-{seed}-output-tokens'] = pred_output_tokens

In [20]:
df[f'{name_model}-{mode}-seed-{seed}-label'].value_counts()

,count
gpt-4o-few-seed-1-label,
0,1271
1,1229


In [21]:
df[f'{name_model}-{mode}-seed-{seed}-latency'].describe()

,gpt-4o-few-seed-1-latency
count,2500.000000
mean,445.808685
std,300.466908
min,254.731469
25%,348.745282
50%,399.299780
75%,466.580333
max,8690.207303


In [22]:
df[f'{name_model}-{mode}-seed-{seed}-ttft'].describe()

,gpt-4o-few-seed-1-ttft
count,2500.000000
mean,443.274048
std,300.382890
min,253.649655
25%,345.875078
50%,396.846106
75%,464.149539
max,8683.976800


In [23]:
df[f'{name_model}-{mode}-seed-{seed}-input-tokens'].describe()

,gpt-4o-few-seed-1-input-tokens
count,2500.000000
mean,612.840000
std,211.559196
min,337.000000
25%,482.000000
50%,537.000000
75%,673.000000
max,1699.000000


In [24]:
df[f'{name_model}-{mode}-seed-{seed}-output-tokens'].describe()

,gpt-4o-few-seed-1-output-tokens
count,2500.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0


# 5. Save Dataset

In [25]:
df.to_csv(path_save, index = False)

# 6. Execution Time

In [26]:
end_notebook = time.time()

In [27]:
delta_notebook = end_notebook - start_notebook
hours_notebook, rem_notebook = divmod(delta_notebook, 3600)
minutes_notebook, seconds_notebook = divmod(rem_notebook, 60)

print(f"Execution Notebook: {int(hours_notebook)}h {int(minutes_notebook)}m {seconds_notebook:.2f}s")

Execution Notebook: 0h 20m 40.79s
